# Query Expansion and Multi-Query Retrieval

Distance-based vector retrieval can fail if the user's phrasing is distinct from the terminology in the document index. Query Expansion prompts an LLM to generate multiple diverse alternative formulations of the user's question, issues parallel vector queries across the index, and consolidates the results via reciprocal ranking and deduplication.

## Workflow Architecture

<div align="center">
  <img src="workflow_query_expansion.png" alt="Query Expansion and Multi-Query Retrieval Architecture Diagram" width="580" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph In["User Query"]
        Q(["1. Original Short Query"]):::startNode
    end
    subgraph Expansion["Multi-Query Expansion"]
        LLM["2. Query Expander<br/><b>Groq gpt-oss-120b</b><br/>(Generates 3-5 Diverse Variations)"]:::llmNode
        V1["Variation 1: Paraphrase"]:::vNode
        V2["Variation 2: Technical Synonyms"]:::vNode
        V3["Variation 3: Specific Sub-Aspect"]:::vNode
    end
    subgraph Search["Parallel Vector Retrieval"]
        Retriever["3. FAISS Retriever<br/><b>HuggingFace MiniLM</b><br/>(Runs Parallel Search per Query)"]:::vsNode
    end
    subgraph Fusion["Reciprocal Rank and Deduplication"]
        RRF["4. Document Fusion and Deduplication<br/>(Merges and Scores Unique Chunks)"]:::fuseNode
    end
    subgraph Out["Final Response"]
        Synth["5. LLM Synthesis Node"]:::synthNode
        Ans(["6. Robust High-Recall Answer"]):::endNode
    end
    Q --> LLM
    LLM --> V1
    LLM --> V2
    LLM --> V3
    V1 --> Retriever
    V2 --> Retriever
    V3 --> Retriever
    Retriever --> RRF
    RRF --> Synth
    Q --> Synth
    Synth --> Ans
    classDef startNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef llmNode fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#0D47A1;
    classDef vNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef vsNode fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef fuseNode fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
    classDef synthNode fill:#FCE4EC,stroke:#C2185B,stroke-width:2px,color:#880E4F;
    classDef endNode fill:#FFEBEE,stroke:#D32F2F,stroke-width:2px,color:#B71C1C;
```
</details>

### Key Retrieval Principles
- **Multi-Perspective Queries**: Generates diverse synonyms and alternate angles to overcome linguistic bias.
- **Parallel Search Execution**: Executes independent vector searches across the MiniLM index.
- **Reciprocal Rank Fusion (RRF)**: Blends candidate rankings to ensure optimal recall without duplicate context.


In [1]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("langchain_crewai_dataset.txt")
raw_doc = loader.load()
raw_doc

C:\Users\itsar\AppData\Local\Temp\ipykernel_45588\4178717238.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\RAG\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content="LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)\nAt the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard patterns like Stuff, Map-Reduce, and Refine. (v1)\nLangChain integrates seamlessly with vector databases like FAISS, Chroma, Pinecone, and Weaviate, enabling semantic search within large document 

In [3]:
from h11._abnf import chunk_size
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_doc)

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-l6-v2")
vectoreStore = FAISS.from_documents(chunks, embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10583.63it/s]


In [5]:
retriver = vectoreStore.as_retriever(search_type='mmr', search_kwargs= {'k':5})
retriver

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001CDD3D3FE00>, search_type='mmr', search_kwargs={'k': 5})

In [8]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="groq:openai/gpt-oss-120b")


In [9]:
llm.invoke('hi')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. It\'s a simple greeting. Should respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 72, 'total_tokens': 111, 'completion_time': 0.082698419, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.002748172, 'prompt_tokens_details': None, 'queue_time': 0.395074586, 'total_time': 0.085446591}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a08ef6-b578-7f02-b754-511d74100709-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 39, 'total_tokens': 111, 'output_token_details': {'reasoning': 21}})

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant
synonyms, technical terms, and useful context.
Original query: "{query}"
Expanded query:
""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain.invoke({'query': 'Langchain Memory'})

'**Expanded query**\n\n```\n"LangChain memory" OR\n"LangChain memory management" OR\n"LangChain conversation memory" OR\n"LangChain stateful memory" OR\n"LangChain memory buffers" OR\n"LangChain memory types" OR\n"LangChain persistent memory" OR\n"LangChain memory API" OR\n"LangChain memory module" OR\n"LangChain memory tutorial" OR\n"LangChain memory examples" OR\n"LangChain memory integration" OR\n"LangChain memory for chatbots" OR\n"LangChain memory architecture" OR\n"LangChain memory caching" OR\n"LangChain memory providers" OR\n"LangChain memory with embeddings" OR\n"LangChain memory vs vector store" OR\n"LangChain memory Python" OR\n"LangChain memory documentation" OR\n"LangChain memory docs" OR\n"LangChain conversation buffer memory" OR\n"ConversationBufferMemory" OR\n"ConversationSummaryMemory" OR\n"VectorStoreRetrieverMemory" OR\n"CombinedMemory" OR\n"ChatMessageHistory" OR\n"LLMChain memory" OR\n"state retention" OR\n"session memory" OR\n"context retention" OR\n"memory store"

In [14]:
from langchain_core.runnables import RunnableMap
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.
Context:
{context}
Question: {input}
""")

document_chain = create_stuff_documents_chain(llm, answer_prompt)

reg_pipeline = (
    RunnableMap({
        'input' : lambda x: x['input'],
        'context' : lambda x: retriver.invoke(query_expansion_chain.invoke({'query': x['input']}))
    }) | document_chain | StrOutputParser()
)

In [15]:
query = {"input": "What types of memory does LangChain support?"}
response = reg_pipeline.invoke(query)
print("Answer:\n", response)

Answer:
 LangChain provides two built‑in memory modules:

1. **ConversationBufferMemory** – stores the full dialogue history so the model can see every previous turn.  
2. **ConversationSummaryMemory** – keeps a running summary of the conversation, letting the model retain context while staying within token limits.  

These are the primary memory types referenced in the documentation.
